# 04 Baselines And Component Checks

Generated by `scripts/revision/create_revision_notebooks.py`.


## Setup

In [ ]:
from pathlib import Path
import json
import os
import subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = DRIVE_ROOT / 'ECG-RAMBA'

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)

def _run_setup(cmd, cwd=None, check=True):
    print(f'$ {cmd}')
    return subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None, check=check)

if (REPO_DIR / '.git').exists():
    os.chdir(REPO_DIR)
    _run_setup('git fetch origin')
    current_branch = subprocess.check_output(
        'git branch --show-current',
        shell=True,
        text=True,
    ).strip()
    if current_branch != BRANCH:
        _run_setup(f'git checkout {BRANCH}')
    _run_setup(f'git pull --ff-only --autostash origin {BRANCH}')
elif (REPO_DIR / 'configs' / 'config.py').exists():
    os.chdir(REPO_DIR)
    print('Repo directory exists but is not a git checkout; skipping git pull.')
elif Path.cwd().joinpath('configs', 'config.py').exists():
    REPO_DIR = Path.cwd()
    os.chdir(REPO_DIR)
    if (REPO_DIR / '.git').exists():
        _run_setup('git fetch origin')
        _run_setup(f'git pull --ff-only --autostash origin {BRANCH}', check=False)
else:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    _run_setup(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)

os.chdir(REPO_DIR)
_run_setup('git status --short --branch', check=False)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)

def run(cmd, check=True):
    print(f'$ {cmd}')
    return subprocess.run(cmd, shell=True, check=check)


def run_script_if_exists(script_path, command):
    path = Path(script_path)
    if path.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')


## Minimum Fair Baseline Matrix

A-level baselines: Full ECG-RAMBA, Raw Mamba, ResNet1D/CNN, MiniRocket-only, HRV-only. All must use the same split, preprocessing, threshold, and aggregation protocol.

In [ ]:
import pandas as pd

registry = pd.read_csv('docs/revision_plan/experiment_registry.csv')
display(registry[registry['experiment_id'].isin(['EXP_BASE', 'EXP_PRED'])])

planned_outputs = [
    'reports/revision/metrics/baseline_summary.csv',
    'reports/revision/tables/table_baselines.csv',
]
for path in planned_outputs:
    print(path, 'exists=', Path(path).exists())


## Placeholder Commands

In [ ]:
RUN_HEAVY = False

planned = {
    'MiniRocket-only': 'python scripts/revision/TBD_minirocket_only.py',
    'HRV-only': 'python scripts/revision/TBD_hrv_only.py',
    'ResNet1D/CNN': 'python scripts/revision/TBD_resnet1d_baseline.py',
    'Raw Mamba': 'python scripts/revision/TBD_raw_mamba_baseline.py',
}

for name, command in planned.items():
    if RUN_HEAVY:
        run(command, check=False)
    else:
        print(f'{name}: planned command -> {command}')
